# Baseline: Linear Regression и CatBoost

**Задача:** предсказать `rented_bike_count` (число аренд за час) в датасете Seoul Bike Sharing.

Валидация — **по времени** (из вывода EDA: сравнение на одинаковых временных фолдах, последний период не трогаем при подборе).

## Imports

In [61]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, train_test_split

from catboost import CatBoostRegressor

import kagglehub

RANDOM_STATE = 42

## Загрузка данных

In [59]:
CSV_PATH = os.path.join(kagglehub.dataset_download("joebeachcapital/seoul-bike-sharing"), "SeoulBikeData.csv")

df = pd.read_csv(CSV_PATH, encoding="unicode_escape")


def norm_col(c: str) -> str:
    c = str(c).strip().lower()
    c = c.replace('°c', '').replace('°', '')
    c = c.replace('%', '')
    c = c.replace('(', '').replace(')', '')
    c = c.replace('/', '_')
    c = re.sub(r'\s+', '_', c)
    c = re.sub(r'_+', '_', c).strip('_')
    return c


df.columns = [norm_col(c) for c in df.columns]
df = df.rename(columns={
    'wind_speed_m_s':        'wind_speed',
    'visibility_10m':        'visibility',
    'dew_point_temperature': 'dew_point_temp',
    'solar_radiation_mj_m2': 'solar_radiation',
    'rainfallmm':            'rainfall',
    'snowfall_cm':           'snowfall',
})
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

df = df.sort_values(['date', 'hour']).reset_index(drop=True)

for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

for col in df.select_dtypes(include=['int64']).columns:
    if col == 'rented_bike_count':
        df[col] = df[col].astype('int16') 
    else:
        df[col] = df[col].astype('int16')

print(df.shape)
print(df.columns.tolist())

(8760, 14)
['date', 'rented_bike_count', 'hour', 'temperature', 'humidity', 'wind_speed', 'visibility', 'dew_point_temp', 'solar_radiation', 'rainfall', 'snowfall', 'seasons', 'holiday', 'functioning_day']


## Признаки и целевая переменная

In [62]:
train_val_df, test_df = train_test_split(
    df, 
    test_size=0.15, 
    shuffle=False,
    random_state=RANDOM_STATE
)

train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.1765,   
    shuffle=False, 
    random_state=RANDOM_STATE
)

features = [c for c in df.columns if c not in ['date', 'rented_bike_count']]

X_train = train_df[features]
y_train = train_df['rented_bike_count']

X_val = val_df[features]
y_val = val_df['rented_bike_count']

X_test = test_df[features]
y_test = test_df['rented_bike_count']

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 6132, Val size: 1314, Test size: 1314


## Построение модели Линейной регрессии

Числовые данные (hour, temperature, dew_point_temp, humidity, wind_speed, visibility, solar_radiation, rainfall, snowfall) масштабируем.
Значения seasons, holiday - OHE.
Значения functioning_day - бинарное (1 или 0).

Целевую переменную и OHE столбцы не масштабируем.

Значения параметра регуляризации Ridge: 1.0

In [63]:
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()

for dataset in [X_train_lr, X_val_lr]:
    dataset['functioning_day'] = dataset['functioning_day'].map({'Yes': 1, 'No': 0}).astype('int8')

num_cols_lr = ['hour', 'temperature', 'dew_point_temp', 'humidity', 
               'wind_speed', 'visibility', 'solar_radiation', 'rainfall', 'snowfall']
cat_cols_lr = ['seasons', 'holiday']
bin_cols_lr = ['functioning_day']

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_lr),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols_lr),
        ('bin', 'passthrough', bin_cols_lr)
    ])

lr_pipeline = make_pipeline(
    preprocessor_lr,
    Ridge(random_state=RANDOM_STATE, alpha=1.0)
)

## Обучение и оценка Линейной регрессии

In [66]:
ts_cv = TimeSeriesSplit(n_splits=5)

cv_scores_lr = cross_val_score(
    lr_pipeline, X_train_lr, y_train,
    cv=ts_cv, scoring='neg_root_mean_squared_error', n_jobs=-1
)

rmse_cv_lr = -cv_scores_lr
print(f"LinearRegression CV RMSE: {rmse_cv_lr.mean():.2f} +/- {rmse_cv_lr.std():.2f}")

lr_pipeline.fit(X_train_lr, y_train)
y_val_pred_lr = lr_pipeline.predict(X_val_lr)

print("\nМетрики Линейной регрессии:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_pred_lr)):.2f}")
print(f"MAE:  {mean_absolute_error(y_val, y_val_pred_lr):.2f}")
print(f"R2:   {r2_score(y_val, y_val_pred_lr):.4f}")

LinearRegression CV RMSE: 487.84 +/- 221.18

Метрики Линейной регрессии:
RMSE: 504.51
MAE:  400.17
R2:   0.5021


C:\Users\andrew\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Построение модели CatBoost

In [57]:
cat_features_cb = ['seasons', 'holiday', 'functioning_day']

cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features_cb]

cb_pipeline = make_pipeline(
    CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        cat_features=cat_features_idx,
        random_state=RANDOM_STATE,
        verbose=0
    )
)

## Обучение и оценка CatBoost

In [65]:
cv_rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(ts_cv.split(X_train), 1):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    cb_pipeline.fit(X_tr, y_tr)
    
    y_pred = cb_pipeline.predict(X_va)
    rmse = np.sqrt(mean_squared_error(y_va, y_pred))
    cv_rmse_scores.append(rmse)

cv_rmse_scores = np.array(cv_rmse_scores)
print(f"\nCatBoost CV RMSE: {cv_rmse_scores.mean():.2f} +/- {cv_rmse_scores.std():.2f}")

cb_pipeline.fit(X_train, y_train)
y_val_pred_cb = cb_pipeline.predict(X_val)

print("\nCatBoost Метрики:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_pred_cb)):.2f}")
print(f"MAE:  {mean_absolute_error(y_val, y_val_pred_cb):.2f}")
print(f"R2:   {r2_score(y_val, y_val_pred_cb):.4f}")


CatBoost CV RMSE: 335.14 +/- 124.58

CatBoost Метрики:
RMSE: 350.46
MAE:  250.01
R2:   0.7597
